In [ ]:
import torch
import einops

from einops.layers.torch import EinMix
from utils.config import *
from utils.components import *
from utils.loss_fn import *

def init_sincos_positions(dim: int, world: WorldConfig):
    # integer indices
    coordinates = torch.stack(torch.unravel_index(indices = torch.arange(world.num_tokens), shape = world.token_shape), dim = -1)
    # log wavelengths
    log_wavelengths = torch.as_tensor(world.token_shape).log()
    # only encode shape dimensions with actual size
    valid = log_wavelengths > 0
    log_wavelengths = log_wavelengths[valid]
    coordinates = coordinates[:, valid]
    # space the frequencies according to the required number of bands
    negative_spacing = torch.linspace(0, -1, dim // (coordinates.size(-1) * 2))
    # calculate the sin/cos embeddings:
    frequencies = torch.exp(negative_spacing * log_wavelengths[..., None])
    angles = torch.einsum("n i, i d -> n i d", coordinates, frequencies) # overflows fp16, be careful
    positions = einops.rearrange([angles.sin(), angles.cos()], 'two n i d -> n (two i d)')
    # avoid uneven dimensions by zero-padding
    positions = torch.nn.functional.pad(positions, (0, dim - positions.size(-1))) 
    return positions

class EinAR(torch.nn.Module):
    def __init__(self, network: NetworkConfig, world: WorldConfig):
        super().__init__()
        # config attributes
        self.network = network
        self.world = world

        # learnable parameters
        self.latent_tokens = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.zeros(network.num_latents, network.dim), std = network.dim ** -0.5))
        self.src_positions = torch.nn.Parameter(init_sincos_positions(network.dim, world= world))

        # I/O
        self.to_tokens = torch.nn.Sequential(
            EinMix(f'b {world.field_pattern} -> b ({world.token_pattern}) c',
                weight_shape = f'v {world.patch_pattern} c', 
                c = default(network.dim_in, network.dim), 
                **world.token_sizes, **world.patch_sizes),
            EinMix(f'b ({world.token_pattern}) c -> b ({world.token_pattern}) d',
                weight_shape = f'v d c',
                d = network.dim, c = default(network.dim_in, network.dim), 
                **world.token_sizes),
            torch.nn.RMSNorm(network.dim)
        )

        self.to_output = torch.nn.Sequential(
            EinMix(f'b ({world.token_pattern}) d -> (k b) {world.field_pattern}',
                   weight_shape = f'k v {world.patch_pattern} d',
                   d = default(network.dim_out, network.dim), k = network.num_tails, 
                   **world.patch_sizes, **world.token_sizes),
            GaussianSmoothing3D(world.field_shape[0], kernel_size= 5, sigma= 1.),
            Rearrange('(k b) ... -> k b ...', k = network.num_tails)
        )
        
        # Encoder
        self.predictor = torch.nn.ModuleList([
                TransformerBlock(dim= network.dim, drop_path= network.drop_path) 
                for _ in range(default(network.num_layers, 1))
                ])
        
        # weight initialization
        self.apply(self.base_init)
        
    def base_init(self, m: torch.nn.Module):
        if isinstance(m, torch.nn.Linear) or isinstance(m, EinMix):
            torch.nn.init.trunc_normal_(m.weight, std = m.weight.size(-1) ** -0.5)
            if exists(m.bias):
                torch.nn.init.zeros_(m.bias)            
   
    def forward(self, fields: torch.FloatTensor, num_steps: int = 1):
        # tokenize input
        src = self.to_tokens(fields) + self.src_positions
        
        # roll-out
        predictions = []
        for _ in range(num_steps):
            # add cls tokens
            cls = einops.repeat(self.latent_tokens, 'z d -> b z d', b= src.size(0))
            latents, shape = einops.pack([src, cls], 'b * d')
            
            # transformer stack
            for predict in self.predictor:
                latents = predict(latents)
            
            # forward src tokens
            src, cls = einops.unpack(latents, shape, 'b * d')
            
            # decode
            pred = self.to_output(src)
            predictions.append(pred)
        
        return einops.pack(predictions, 'k b v * h w')[0]

In [51]:
network = NetworkConfig(dim = 384, num_latents= 16, num_layers= 8, drop_path= 0.1, num_tails= 2)
world = WorldConfig({'v': 3, 't': 6, 'h': 64, 'w': 120}, patch_sizes= {'vv': 3, 'tt': 1, 'hh': 8, 'ww': 8}, batch_size=4)

In [52]:
ar = EinAR(world=world, network=network)

In [53]:
fields = torch.randn((world.batch_size, *world.field_shape))

In [57]:
test = ar(fields, 4)

In [58]:
test.shape

torch.Size([2, 4, 3, 24, 64, 120])

In [59]:
world.num_tokens

720